In [ ]:
# Import required packages

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
from shapely.geometry import Point
from numpy import trapz
from scipy.stats import gamma,norm,fisk,wilcoxon
from sklearn.cluster import KMeans
import sys
from pathlib import Path
import logging
from matplotlib.patches import Patch

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.append(str(project_root))


: 

In [2]:
# Configuration
config = {
    'casr_input_data': project_root / 'data' / 'input_data' / 'CaSR_SWE' / 'bow_casrv3.2_SWE_P_data.csv',
    'SWEI_output_dir': project_root / 'data' / 'output_data' / 'for_paper'/'SWEI',
    'SWEI_plots': project_root / 'data' / 'output_plots' / 'for_paper'/'SWEI',
}

In [3]:
# Load CaSR data
casr_data = pd.read_csv(config['casr_input_data'])
casr_data['time'] = pd.to_datetime(casr_data['time'])
# rename columns for consistency
casr_data.rename(columns={'Grid_id': 'station_id'}, inplace=True)
display(casr_data.head())

,station_id,time,lon,lat,Precipitation,SWE,Elevation_Category
0,1,1980-10-10,-116.1380,51.3346,0.009879,0.878418,2000_2500m
1,1,1982-07-19,-116.1380,51.3346,2.765854,0.515625,2000_2500m
2,2,1983-10-12,-116.1875,51.4191,0.023846,6.875000,2000_2500m
3,2,1980-12-11,-116.1875,51.4191,4.578907,118.125000,2000_2500m
4,2,1981-07-21,-116.1875,51.4191,5.400473,0.000000,2000_2500m


## Select data for the season

In [4]:
# Enforce allowed seasonal-year range inside the selector
def select_seasonal_data(ts, start_month, end_month, min_year, max_year):
    month = ts.month
    year = ts.year
    if month >= start_month:
        seasonal_year = year
    elif month <= end_month:
        seasonal_year = year - 1
    else:
        return np.nan
    return seasonal_year if (min_year <= seasonal_year <= max_year) else np.nan

In [5]:
# For CaSR data
casr_data['Seasonal_Year'] = casr_data['time'].apply(
    lambda ts: select_seasonal_data(ts, start_month=10, end_month=5, min_year=1980, max_year=2023)
)
# Remove rows where Seasonal_Year is NaN (outside the allowed data range)
casr_data = casr_data.dropna(subset=['Seasonal_Year'])

# rename Grid_id to station_id in casr_data
casr_data.rename(columns={'Grid_id': 'station_id'}, inplace=True)

#save the final CaSR data with Seasonal_Year column
#casr_data.to_csv(output_dir / 'bow_casr_data_with_seasonal_year.csv', index=False)

# Display final CaSR data
display(casr_data)

,station_id,time,lon,lat,Precipitation,SWE,Elevation_Category,Seasonal_Year
0,1,1980-10-10,-116.1380,51.3346,0.009879,0.878418,2000_2500m,1980.0
2,2,1983-10-12,-116.1875,51.4191,0.023846,6.875000,2000_2500m,1983.0
3,2,1980-12-11,-116.1875,51.4191,4.578907,118.125000,2000_2500m,1980.0
5,2,1982-10-29,-116.1875,51.4191,1.777088,23.125000,2000_2500m,1982.0
6,2,1982-01-18,-116.1875,51.4191,1.890419,145.000000,2000_2500m,1981.0
...,...,...,...,...,...,...,...,...
4224297,257,2024-02-22,-111.6202,50.0960,0.011238,6.250000,500_1000m,2023.0
4224298,257,2024-03-19,-111.6202,50.0960,0.000257,0.000488,500_1000m,2023.0
4224301,257,2024-01-06,-111.6202,50.0960,0.135581,0.625000,500_1000m,2023.0
4224302,257,2024-05-17,-111.6202,50.0960,6.328805,0.000000,500_1000m,2023.0


## SWEI

In [6]:
# Functions for SWEI calculation
def extract_grid_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract per-Grid static metadata.
    """
    return (
        df[["station_id", "lon", "lat", "Elevation_Category"]]
        .drop_duplicates("station_id")
        .set_index("station_id")
    )

def perturb_daily_swe_zeros(
    df: pd.DataFrame,
    swe_col: str = "SWE",
    id_col: str = "station_id",
    seed: int = 42,
    perturb_factor: float = 0.01
) -> pd.DataFrame:
    """
    Perturb exact daily SWE zeros before integration.
    Keeps station_id as a normal column.
    """

    out = df.copy().reset_index(drop=True)

    if id_col not in out.columns:
        raise KeyError(f"{id_col} is not in columns. Columns are: {out.columns.tolist()}")

    out[swe_col] = out[swe_col].astype(float)

    rng = np.random.default_rng(seed)

    for sid, idx in out.groupby(id_col).groups.items():

        vals = out.loc[idx, swe_col]

        valid = vals.notna()
        zero_idx = vals[valid & (vals == 0)].index
        positive_vals = vals[valid & (vals > 0)]

        if len(zero_idx) == 0:
            continue

        if positive_vals.empty:
            continue

        min_positive = positive_vals.min()

        out.loc[zero_idx, swe_col] = rng.uniform(
            low=np.nextafter(0, 1),
            high=min_positive * perturb_factor,
            size=len(zero_idx)
        )

    return out

def daily_to_monthly_swe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Daily perturbed SWE → monthly integrated SWE.
    Seasonal_Year is recomputed from time.
    """

    out = df.copy()
    out["time"] = pd.to_datetime(out["time"])

    monthly = (
        out
        .groupby(
            ["station_id", pd.Grouper(key="time", freq="MS")],
            as_index=False
        )
        .agg(
            SWE_monthly=("SWE_perturbed", "sum")
        )
    )

    # Recompute Seasonal_Year (e.g., Oct–Sep water year)
    monthly["Seasonal_Year"] = np.where(
        monthly["time"].dt.month >= 10,
        monthly["time"].dt.year,
        monthly["time"].dt.year - 1
    )
    
    return monthly


def rolling_integrated_swe_by_season(
    monthly_df: pd.DataFrame,
    window_months: int
) -> pd.DataFrame:
    """
    Compute rolling k‑month integrated SWE within each Seasonal_Year.

    • Rolling windows do NOT cross Seasonal_Year boundaries.
    • First (k‑1) months of each season are dropped.
    • Works for any window (3, 6, 8, …).
    """

    out = monthly_df.copy()
    out = out.sort_values(["station_id", "Seasonal_Year", "time"])

    out[f"SWE_{window_months}mo"] = (
        out
        .groupby(["station_id", "Seasonal_Year"])["SWE_monthly"]
        .rolling(window=window_months, min_periods=window_months)
        .sum()
        .reset_index(level=[0, 1], drop=True)
    )

    return out.dropna(subset=[f"SWE_{window_months}mo"])



def gringorten_probabilities(x: np.ndarray) -> np.ndarray:
    """
    Gringorten plotting position with:
    - NaN handling
    - average ranks for ties
    - probability clipping
    """
    x = np.asarray(x, float)
    out = np.full_like(x, np.nan)

    mask = ~np.isnan(x)
    xv = x[mask]

    if xv.size == 0:
        return out

    # ranks with average ties
    order = np.argsort(xv, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(xv) + 1)

    uvals, inv, cnt = np.unique(xv, return_inverse=True, return_counts=True)
    for i, c in enumerate(cnt):
        if c > 1:
            idx = np.where(inv == i)[0]
            ranks[idx] = ranks[idx].mean()

    N = float(len(xv))
    p = (ranks - 0.44) / (N + 0.12)
    p = np.clip(p, 1e-12, 1 - 1e-12)

    out[mask] = p
    return out


def compute_swei_for_grid(
    df: pd.DataFrame,
    swe_col: str
) -> pd.DataFrame:
    """
    Compute SWEI for ONE station/grid using calendar-month standardization.
    """

    out = df.copy()
    out["month"] = out["time"].dt.month

    pvals = np.full(len(out), np.nan)
    zvals = np.full(len(out), np.nan)

    for m in range(1, 13):
        idx = out["month"] == m
        vals = out.loc[idx, swe_col]

        if vals.notna().sum() == 0:
            continue

        p = gringorten_probabilities(vals.values)
        z = norm.ppf(p)

        pvals[idx] = p
        zvals[idx] = z

    out["Gringorten_p"] = pvals
    out["SWEI"] = zvals

    return out



def compute_swei(
    df: pd.DataFrame,
    window_months: int = 3,
    seed: int = 42
) -> pd.DataFrame:
    """
    End-to-end SWEI calculation with daily zero perturbation
    before monthly and rolling integration.
    """

    # 0. Extract static metadata
    grid_meta = extract_grid_metadata(df)

    # 1. Prepare daily data
    daily = df.copy()
    daily["time"] = pd.to_datetime(daily["time"])

    # 2. Create perturbed SWE column
    daily["SWE_perturbed"] = daily["SWE"]

    daily = perturb_daily_swe_zeros(
    daily,
    swe_col="SWE_perturbed",
    id_col="station_id",
    seed=seed,
    perturb_factor=0.01
    )

    # 3. Daily → monthly integrated SWE
    monthly = daily_to_monthly_swe(daily)

    # 4. Rolling monthly integration
    integ = rolling_integrated_swe_by_season(monthly, window_months)

    # 5. Compute SWEI per station/grid
    swei = (
        integ
        .groupby("station_id", group_keys=False)
        .apply(
            lambda g: compute_swei_for_grid(
                g,
                swe_col=f"SWE_{window_months}mo"
            )
        )
        .reset_index(drop=True)
    )
    if "station_id" not in swei.columns:
        swei["station_id"] = integ["station_id"].values    
    print(swei.columns.tolist())
    print(swei.head())

    # 6. Reattach static metadata
    swei = swei.reset_index()

    swei = swei.merge(
    grid_meta.reset_index(),
    on="station_id",
    how="left"
)

    return swei

In [7]:
swei_8mo = compute_swei(casr_data, window_months=8, seed=42)
display(swei_8mo.head())

['time', 'SWE_monthly', 'Seasonal_Year', 'SWE_8mo', 'month', 'Gringorten_p', 'SWEI', 'station_id']
        time  SWE_monthly  Seasonal_Year       SWE_8mo  month  Gringorten_p  \
0 1981-05-01  5072.800764           1980  30306.975816      5      0.556664   
1 1982-05-01  7276.954120           1981  34493.079364      5      0.737987   
2 1983-05-01  5607.820307           1982  34208.649527      5      0.692656   
3 1984-05-01  5777.537110           1983  28006.298830      5      0.420671   
4 1985-05-01  4583.076164           1984  26935.667961      5      0.352675   

       SWEI  station_id  
0  0.142516           1  
1  0.637153           1  
2  0.503394           1  
3 -0.200177           1  
4 -0.378110           1  


,index,time,SWE_monthly,Seasonal_Year,SWE_8mo,month,Gringorten_p,SWEI,station_id,lon,lat,Elevation_Category
0,0,1981-05-01,5072.800764,1980,30306.975816,5,0.556664,0.142516,1,-116.138,51.3346,2000_2500m
1,1,1982-05-01,7276.954120,1981,34493.079364,5,0.737987,0.637153,1,-116.138,51.3346,2000_2500m
2,2,1983-05-01,5607.820307,1982,34208.649527,5,0.692656,0.503394,1,-116.138,51.3346,2000_2500m
3,3,1984-05-01,5777.537110,1983,28006.298830,5,0.420671,-0.200177,1,-116.138,51.3346,2000_2500m
4,4,1985-05-01,4583.076164,1984,26935.667961,5,0.352675,-0.378110,1,-116.138,51.3346,2000_2500m
